# Geometric EEG SSL v2 — Cross-Montage Pretrain Notebook (GPU)

**Purpose:** Pretrain v2 with mixed cross-montage corpus, three
leave-one-dataset-out runs. Headline experiment for v2.

**Prereq:** Run `colab_download.ipynb` first to cache all three datasets
(PhysioNet MI, BCIC-2B, Sleep-EDFx) and their preprocessed signal arrays.
v2 reuses v1's signal cache unchanged — only `ch_pos` is recomputed
on the fly via `src/v2/preprocess.ch_pos_from_names` (fixed-scale,
shared across montages).

Each pretrain cell auto-resumes from the latest checkpoint if interrupted.


## 1. Install dependencies

In [1]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy


## 2. Mount Google Drive + paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data dir (raw downloads from colab_download.ipynb)
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.environ['MNE_DATA'] = MNE_DATA_DIR

# v1 signal cache; reused unchanged by v2.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
if not os.path.isdir(CACHE_ROOT):
    print(f'WARNING: no cache at {CACHE_ROOT}. Loaders will reprocess; '
          'run colab_download.ipynb section 5 to build the cache once.')
else:
    print(f'Reusing v1 signal cache at {CACHE_ROOT}')

# v2 checkpoints go to a dedicated subtree so they don't shadow v1's.
CKPT_ROOT = f'{DRIVE_ROOT}/runs/pretrain'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Checkpoints -> {CKPT_ROOT}')


Mounted at /content/drive
Reusing v1 signal cache at /content/drive/MyDrive/geometric_eeg_ssl/cache
Checkpoints -> /content/drive/MyDrive/geometric_eeg_ssl/runs/pretrain


## 2b. Keep Colab alive

Colab sessions die when the browser tab loses focus or the laptop sleeps.
Checkpoints saved every 10 epochs keep work recoverable, but the keep-alive
ping below buys uninterrupted long runs. Re-run after any browser refresh.


In [3]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed.')


<IPython.core.display.Javascript object>

Keep-alive armed.


## 3. Clone repo (v2-improvements branch)

In [4]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone -b v2-improvements https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} checkout v2-improvements && git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR, '(branch v2-improvements)')


Cloning into '/content/geometric-eeg-ssl'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (314/314), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 314 (delta 152), reused 233 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (314/314), 935.24 KiB | 21.25 MiB/s, done.
Resolving deltas: 100% (152/152), done.
Repo ready at /content/geometric-eeg-ssl (branch v2-improvements)


## 4. Verify the v2 stack imports + smoke

Quick sanity check the v2 modules import cleanly. If this fails, every
pretrain cell below will too -- fix imports first.


In [5]:
import subprocess
out = subprocess.run(
    ['python', f'{REPO_DIR}/tests/smoke_test_v2.py'],
    cwd=REPO_DIR, capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(out.stdout)
if out.returncode != 0:
    print('STDERR:', out.stderr)
    raise RuntimeError('v2 smoke test failed')


smoke OK; 4 mixed-corpus steps, losses[-1]=('bcic_2b', 0.7913130521774292, -0.28106310963630676, 1.0723761320114136)
zero-shot to held-out sleep_edfx (M=2): backbone output shape (2, 2, 16, 256)



## 4b. Build v2 caches locally

v2 uses 9 subjects per dataset (Option A balanced design; see
`docs/v2/experiment_protocol.md`). The 9-subject caches are small
(50 MB phys, 20 MB bcic, ~700 MB sleep) so we build them straight to
local disk in one pass -- no Drive copy round-trips, no FUSE retries,
no OOM risk.

~15 minutes total on first run. Cached locally for the session;
subsequent cells in the same session skip the rebuild.


In [6]:
import os, sys
LOCAL_CACHE = '/content/cache_local'
os.makedirs(LOCAL_CACHE, exist_ok=True)
os.environ['EEG_CACHE_DIR'] = LOCAL_CACHE

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
from src.config import Config
from src.datasets.physionet_mi import PhysioNetMI, EXCLUDED_SUBJECTS
from src.datasets.bcic_2b import BCIC2B, ALL_SUBJECTS as BCIC_ALL
from src.datasets.sleep_edfx import SleepEDFx, ALL_SUBJECTS as SLEEP_ALL

N = 9  # subjects per dataset; matches scripts/v2_pretrain.py default
cfg = Config()

phys_subj  = [s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS][:N]
bcic_subj  = list(BCIC_ALL)[:N]
sleep_subj = list(SLEEP_ALL)[:N]

print(f'EEG_CACHE_DIR -> {LOCAL_CACHE}')
print(f'building v2 caches for {N} subjects/dataset...')
PhysioNetMI(subjects=phys_subj,  cfg=cfg, mode='pretrain', verbose=True).load()
BCIC2B    (subjects=bcic_subj,   cfg=cfg, mode='pretrain', verbose=True).load()
SleepEDFx (subjects=sleep_subj,  cfg=cfg, mode='pretrain', verbose=True).load()
print('done')


EEG_CACHE_DIR -> /content/cache_local
building v2 caches for 9 subjects/dataset...
[cache] building physionet_mi/pretrain; will save to physionet_mi_pretrain_a622abb6599d45f0.npz


FileNotFoundError: Download location /content/drive/MyDrive/geometric_eeg_ssl/mne_data as specified by MNE_DATA does not exist. Either create this directory manually and try again, or set MNE_DATA to an existing directory.

## 5. Verify all three datasets are cached

v2 needs all three for the three leave-one-out splits. If any is missing,
run the corresponding section of `colab_download.ipynb`.


In [ ]:
import os
files = sorted(os.listdir(LOCAL_CACHE)) if os.path.isdir(LOCAL_CACHE) else []
needed = ['physionet_mi_pretrain_', 'bcic_2b_pretrain_', 'sleep_edfx_pretrain_']
missing = [w for w in needed if not any(f.startswith(w) for f in files)]
for w in needed:
    present = [f for f in files if f.startswith(w)]
    print(f'  {w:30s}  {present[0] if present else "MISSING"}')
if missing:
    raise RuntimeError(f'Missing cache prefixes: {missing}. Re-run section 4b.')
print('All three v2 caches present.')


## 6. Per-dataset budget

The three datasets have very different epoch counts:

| Dataset | Epochs available | Channels |
|---|---|---|
| PhysioNet MI | ~9,450 | 64 |
| BCIC-2B | ~6,500 | 3 |
| Sleep-EDFx | ~1,000,000 | 2 (bipolar) |

The v2.2 claim is "distribution of g_ij matters, count doesn't" -- so we cap
each dataset at the same budget per pass. Small datasets are oversampled
with replacement; Sleep-EDFx is subsampled hard.

`EPOCHS_PER_DATASET = 8000` gives roughly the same wall-clock per pass as v1
single-dataset pretraining (which ran ~9,450 PhysioNet epochs/epoch). Bump
or lower as compute allows.


In [ ]:
EPOCHS_PER_DATASET = 4096  # per dataset per pass; must be a multiple of batch_size (64). 4096 = 64*64.
N_SUBJECTS_PER_DATASET = 9
print(f'budget per dataset per pass = {EPOCHS_PER_DATASET}')
print(f'subjects per dataset = {N_SUBJECTS_PER_DATASET}')


## 7.geometric. Pretrain v2 -- variant `geometric` (g3 + 10-D descriptor + per-channel target)

Three leave-one-out runs. Each cell is independent; you can run them serially or interleave with the other variant.


#### v2_geometric_no_sleep  (Sleep-EDFx held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_geometric_no_sleep'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_geometric_no_sleep: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant geometric \
    --pretrain-datasets physionet_mi,bcic_2b \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_geometric_no_sleep.txt


#### v2_geometric_no_bcic  (BCIC-2B held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_geometric_no_bcic'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_geometric_no_bcic: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant geometric \
    --pretrain-datasets physionet_mi,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_geometric_no_bcic.txt


#### v2_geometric_no_phys  (PhysioNet MI held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_geometric_no_phys'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_geometric_no_phys: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant geometric \
    --pretrain-datasets bcic_2b,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_geometric_no_phys.txt


## 7.chind. Pretrain v2 -- variant `chind` (no geometry; channel-independent baseline)

Three leave-one-out runs. Each cell is independent; you can run them serially or interleave with the other variant.


#### v2_chind_no_sleep  (Sleep-EDFx held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_chind_no_sleep'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_chind_no_sleep: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant chind \
    --pretrain-datasets physionet_mi,bcic_2b \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_chind_no_sleep.txt


#### v2_chind_no_bcic  (BCIC-2B held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_chind_no_bcic'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_chind_no_bcic: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant chind \
    --pretrain-datasets physionet_mi,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_chind_no_bcic.txt


#### v2_chind_no_phys  (PhysioNet MI held out)


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_chind_no_phys'
os.makedirs(CKPT_DIR, exist_ok=True)
_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_chind_no_phys: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --variant chind \
    --pretrain-datasets bcic_2b,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --n-subjects-per-dataset {N_SUBJECTS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_v2_chind_no_phys.txt


## 7.codex. Pretrain v2 -- variant `codex` (DEFERRED)

Per-montage transductive codex. **Not yet implemented.** Open design questions:
1. Per-montage codex tables (3 separate, switched per batch) vs. unified.
2. Held-out fallback: random vs. spatial nearest-neighbor.
3. Whether to fine-tune new codex slots at probe time (blurs zero-shot claim).

See `docs/v2/experiment_protocol.md` for the protocol once decided. Cells will be added here when implemented.


## 8. Checkpoint inventory


In [ ]:
import glob, os

variants = ['geometric', 'chind']  # codex added later
splits = ['no_sleep', 'no_bcic', 'no_phys']
for v in variants:
    for s in splits:
        d = f'{CKPT_ROOT}/v2_{v}_{s}'
        ckpts = sorted(glob.glob(f'{d}/epoch_*.pt'))
        last = ckpts[-1].split('/')[-1] if ckpts else 'NONE'
        print(f'v2_{v:10s}_{s:9s}  ckpts={len(ckpts):3d}  latest={last}')


## Done

Six v2 pretrain runs complete (geometric + chind, three splits each).
Codex variant is deferred -- see section 7.codex.

Next: the v2 experiment notebook (to be written) runs probe evaluation
on the held-out dataset for each checkpoint, per `docs/v2/experiment_protocol.md`.
